### Setup Notebook

In [7]:
import os
import re

import json
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI

In [8]:
LLM_MODEL = 'gemma2:2b'

### Utility Functions

In [9]:
def export_df_to_json(df, filename):
    '''
    df : the dataframe that needs to be dumped as json
    filename : string path and name of the destination json file
    '''
    df_json = df.to_dict(orient="records")
    with open(filename, 'wt') as f_out:
        json.dump(df_json, f_out, indent=2)



def load_jsonfile(filename):
    '''
    filename : filepath and filename(.json file) as a single string
    This function can be used to read any json file
    '''
    with open(filename, 'rt') as f:
        data_json = json.load(f)
    return data_json

In [10]:
def find_symbol_positions(input_string):

    positions = {'[': [], ']': [], '"': [], "'": [], '-': [], '?': []}

    for symbol in positions.keys():
        # find all matches of the current symbol in the input string
        matches = [match.start() for match in re.finditer(re.escape(symbol), input_string)]
        positions[symbol] = matches

    return positions



def extract_query5(input_string, positions):
    if (len(positions['['])==0) or (len(positions[']'])==0):
        # if not within box brackets, queries might be between '-' and '?'
        quoted_strings = []
        j = 0
        for dash in positions['-']:
            while j < len(positions['?']) and positions['?'][j] <= dash:
                j+=1
            if j < len(positions['?']):
                quoted_strings.append(input_string[dash+1 : positions['?'][j]+1].replace('*','').strip())
        return quoted_strings
    
    start_bracket = positions['['][0]
    end_bracket = next((pos for pos in positions[']'] if pos > start_bracket), None)

    if end_bracket is None:
        return []
    
    quoted_strings = []
    quote_positions = [pos for pos in positions['"'] if start_bracket < pos < end_bracket]
    single_quotes = positions["'"]

    for i in range(0, len(quote_positions) - 1, 2):
        start_quote = quote_positions[i]
        end_quote = quote_positions[i + 1]
        for sq in single_quotes:
            if start_quote < sq < end_quote:
                single_quotes.remove(sq)
        # extract the string between quotes and add it to the list
        quoted_strings.append(input_string[start_quote + 1:end_quote].replace('*','').strip())
    
    if (quoted_strings==[]) and (len(positions['?']) > 0) and (len(single_quotes) > 0):
        # searching for queries be between "'" and '?'
        j = 0
        for squote in positions['-']:
            while j < len(single_quotes) and positions['?'][j] <= squote:
                j+=1
            if j < len(positions['?']):
                quoted_strings.append(input_string[squote+1 : positions['?'][j]+1].replace('*','').strip())

    return quoted_strings

In [11]:
def llm(prompt, llm_model='gemma2:2b'):
    '''
    This function uses 'llm_model' to generate response for the provided input 'prompt' to llm.
    '''
    client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',
    )
    
    response = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        # temperature=0,    # remove randomness for deterministic output but not using it as it makes summary clumsy with phrases like 'you stated correctly...', 'you explained it well...' etc.
        seed=72
    )

    return response.choices[0].message.content

In [12]:
def generate_questions(tscript_chunk, llm_model='gemma2:2b'):
    '''
    This function takes in a 'transcript' text and generates summarized text using 'llm_model' specified.
    '''
    querygen_prompt = """You emulate a student who is learning from youtube videos.
        Formulate 5 questions this user might ask based on a provided video transcript, cleaned and summarized texts and its keywords.
        Make the questions specific to this transcript.
        The record should contain the answer to the questions, and the questions should be complete and not too short.
        Use as fewer words as possible from the record.

        The record:

        original_transcript: {text}
        summary_text: {smry_text}
        cleaned_text: {clean_text}
        keywords: {keywords}

        Provide the output in a python list without using code blocks:

        ["question1", "question2", ..., "question5"]
        """.strip()

    prompt = querygen_prompt.format(**tscript_chunk)

    return llm(prompt, llm_model)

In [13]:
def generate_query_file(documents, llm_model='gemma2:2b', retry=3, debug=0):
    '''
    Iterates through each row and generates 5 possible queries a viewer/student could ask.
    since LLMs are unreliable, we try 'retry' number of times to generate queries before passing on to next chunk.
    Returns a dataframe with query column and uid to identify the ground truth chunk.
    '''
    print(f"INFO: initiated query generation")
    print(f"INFO: total text chunks {len(documents)}")
    
    groundtruth = []
    success=0
    failure=0

    for doc in tqdm(documents, total=len(documents), desc="Generating Queries"):
        attempt=1
        query5=[]

        while(query5==[] and attempt<=retry):
            llm_response = generate_questions(doc, llm_model)
            positions = find_symbol_positions(llm_response)
            query5 = extract_query5(llm_response, positions)
            attempt+=1

        for query in query5:
            temp = {'uid': doc['uid'], 'query':query}
            groundtruth.append(temp)
            if debug:
                print(f"Query: {query}\n")
        
        if query5==[]:
            failure+=1
        else:
            success+=1
    
    df_query5 = pd.DataFrame(groundtruth)

    print(f"INFO: query generation finished")
    print(f"INFO: total documents - {len(documents)} | success - {success} | failure - {failure}")
    return df_query5

### Import Chunked Transcribe Data

In [14]:
tscribe_data_loc = "../app/app_data/multi_tscribe.json"
tscribe_data = pd.read_json(tscribe_data_loc)
tscribe_data['video_id'] = tscribe_data['uid'].apply(lambda x: x.split('__B')[0])
tscribe_data.head(3)

,uid,text,smry_text,clean_text,keywords,video_id
0,t6Wc7OMks4U__B1__S0.0,"Every piece of art tells a story. Usually, mo...","Biombo: A Folding Screen \nJapanese design, d...","Every piece of art tells a story. Usually, mor...","creative process, art history, biographies, \n...",t6Wc7OMks4U
1,t6Wc7OMks4U__B2__S61.88,"you look at, and who’s doing the telling. By ...",### Crash Course Art History: Unveiling art's ...,"""You look at and who’s telling it. By studying...","sart history, art appreciation, object studyin...",t6Wc7OMks4U
2,t6Wc7OMks4U__B3__S121.84,but also reveals the ideas that \nconnect peo...,This transcript explores the history of Art Hi...,And also reveals ideas that connect people acr...,"Keyword1: Art History, #Keyword 2: Renaissance...",t6Wc7OMks4U


In [15]:
tscribe_data.shape, tscribe_data['video_id'].unique()

((196, 6),
 array(['t6Wc7OMks4U', 'zjkBMFhNj_g', '1A_CAkYt3GY', 'NKEhdsnKKHs',
        'AUXIuYHFgBE', 'YQcuTYcxxWc', '-wrCpLJ1XAw', 'IV-8YsyghbU',
        'MLKrmw906TM', '5C-s4JrymKM', 'kXhJ3hHK9hQ', '-X8Xfl0JdTQ',
        'FmTsS5xFA6k', 'TgisehuGOyY', '7e9v_fsZB6A', 'gs_gY1K1AMU',
        '9AzNEG1GB-k', 'AYkhlXronNk', 'S93jMOqF-oE', 'YaDvRdLMkHs',
        'mjQwedC1WzI', '-TFCMK4i2lo', 'trqDnLNRuSc', '17WiQ_tNld4'],
       dtype=object))

In [16]:
documents = load_jsonfile("../app/app_data/multi_tscribe.json")

len(documents), documents[0]

(196,
 {'uid': 't6Wc7OMks4U__B1__S0.0',
  'text': " Every piece of art tells a story. Usually, more than one. I’ll show you. This is a biombo, a folding screen made in Mexico\xa0\nand inspired by Japanese decorative designs. It existed before we had pianos\xa0\nor bicycles or even sandwiches. This one would have decorated a wealthy Spaniard’s\xa0\nhome in late seventeenth-century Mexico. And while it might look like mere decoration,\xa0\nif you lean in, it'll tell you a story. A story of a battle, where Spain conquered\xa0\nTenochtitlan, or modern day Mexico City,\xa0\xa0 and the Aztec emperor Moctezuma the Second\xa0\nwas killed by one of his own subjects. But records from the Indigenous people of\xa0\nMexico tell a different story of the same event. In their account, the Spanish killed the\xa0\nAztec emperor, as part of their conquest. Historical events like this — and history\xa0\nitself — aren’t exactly straightforward. The story you get varies, depending\xa0\non which documents, o

### Generate Sample Queries

In [95]:
def find_symbol_positions(input_string):

    positions = {'[': [], ']': [], '"': [], "'": [], '-': [], '?': []}

    for symbol in positions.keys():
        # find all matches of the current symbol in the input string
        matches = [match.start() for match in re.finditer(re.escape(symbol), input_string)]
        positions[symbol] = matches

    return positions



def extract_query5_v2(input_string, positions, threshold=20):
    # threshold is just a failsafe against short quoted words
    if (len(positions['['])==0) or (len(positions[']'])==0):
        # if not within box brackets, queries might be between '-' and '?'
        quoted_strings = []
        j = 0
        for dash in positions['-']:
            while j < len(positions['?']) and positions['?'][j] <= dash:
                j+=1
            if j < len(positions['?']):
                if len(input_string[dash+1 : positions['?'][j]+1].replace('*','').strip()) > threshold:
                    quoted_strings.append(input_string[dash+1 : positions['?'][j]+1].replace('*','').strip())
        return quoted_strings
    
    quoted_strings = []

    for i in range(len(positions['['])):
        quoted_strings_tmp = []
        start_bracket = positions['['][i]
        end_bracket = next((pos for pos in positions[']'] if pos > start_bracket), None)

        if end_bracket is None:
            break
        
        quote_positions = [pos for pos in positions['"'] if start_bracket < pos < end_bracket]
        single_quotes = [pos for pos in positions["'"] if start_bracket < pos < end_bracket]

        print("DEBUG : quote_positions, single_quotes -- ", quote_positions, single_quotes)
        for i in range(0, len(quote_positions) - 1, 2):
            start_quote = quote_positions[i]
            end_quote = quote_positions[i + 1]
            for sq in single_quotes:
                if start_quote < sq < end_quote:
                    single_quotes.remove(sq)
            # extract the string between quotes and add it to the list
            if len(input_string[start_quote + 1:end_quote].replace('*','').strip()) > threshold:
                quoted_strings_tmp.append(input_string[start_quote + 1:end_quote].replace('*','').strip())
        
        if (quoted_strings_tmp==[]) and (len(positions['?']) > 0) and (len(single_quotes) > 0):
            # searching for queries be between "'" and '?'
            j = 0
            covered_pos = 0
            for squote in single_quotes:
                if covered_pos >= squote:
                    continue
                while j < len(positions['?']) and positions['?'][j] <= squote:
                    j+=1
                if j < len(positions['?']):
                    print("DEBUG: squote, j -- ",squote,j)
                    if len(input_string[squote+1 : positions['?'][j]+1].replace('*','').strip()) > threshold:
                        quoted_strings_tmp.append(input_string[squote+1 : positions['?'][j]+1].replace('*','').strip())
                        covered_pos = min([x for x in positions["'"] if x>positions['?'][j]]) # replaced 'single_quotes with positions["'"] to consider cases with "'" outside box bracket
                        # as add on case "'" could be before box bracket as well and this will lead to incorrect query getting captured
                        print("DEBUG: single_quotes, j, questionmark, covered_pos -- ", single_quotes, j, positions['?'][j], covered_pos)

        quoted_strings.extend(quoted_strings_tmp)

    return quoted_strings

In [ ]:
def find_symbol_positions(input_string):

    positions = {'[': [], ']': [], '"': [], "'": [], '-': [], '?': []}

    for symbol in positions.keys():
        # find all matches of the current symbol in the input string
        matches = [match.start() for match in re.finditer(re.escape(symbol), input_string)]
        positions[symbol] = matches

    return positions



def extract_query5_v3(input_string, positions, threshold=20):
    # threshold is just a failsafe against short quoted words
    if (len(positions['['])==0) or (len(positions[']'])==0):
        # if not within box brackets, queries might be between '-' and '?'
        quoted_strings = []
        j = 0
        for dash in positions['-']:
            while j < len(positions['?']) and positions['?'][j] <= dash:
                j+=1
            if j < len(positions['?']):
                if len(input_string[dash+1 : positions['?'][j]+1].replace('*','').strip()) > threshold:
                    quoted_strings.append(input_string[dash+1 : positions['?'][j]+1].replace('*','').strip())
        return quoted_strings
    
    quoted_strings = []

    for i in range(len(positions['['])):
        quoted_strings_tmp = []
        start_bracket = positions['['][i]
        end_bracket = next((pos for pos in positions[']'] if pos > start_bracket), None)

        if end_bracket is None:
            break
        
        double_quotes = [pos for pos in positions['"'] if start_bracket < pos < end_bracket]
        single_quotes = [pos for pos in positions["'"] if start_bracket < pos < end_bracket]
        all_quotes = sorted(double_quotes + single_quotes)

        print("DEBUG : double_quotes, single_quotes -- ", double_quotes, single_quotes)
        
        if (len(positions['?']) > 0) and (len(all_quotes) > 0):
            # searching for queries be between quotes and '?'
            j = 0
            covered_pos = 0
            for quote in all_quotes:
                if covered_pos >= quote:
                    continue
                while j < len(positions['?']) and positions['?'][j] <= quote:
                    j+=1
                if j < len(positions['?']):
                    print("DEBUG: quote, j -- ",quote,j)
                    if len(input_string[quote+1 : positions['?'][j]+1].replace('*','').strip()) > threshold:
                        quoted_strings_tmp.append(input_string[quote+1 : positions['?'][j]+1].replace('*','').strip())
                        covered_pos = min([x for x in positions["'"] if x>positions['?'][j]]) # replaced 'single_quotes with positions["'"] to consider cases with "'" outside box bracket
                        # as add on case "'" could be before box bracket as well and this will lead to incorrect query getting captured
                        print("DEBUG: single_quotes, j, questionmark, covered_pos -- ", single_quotes, j, positions['?'][j], covered_pos)

        quoted_strings.extend(quoted_strings_tmp)

    return quoted_strings

In [99]:
# sample queries for a single document/chunk
id=1
querydoc = generate_questions(documents[id], llm_model=LLM_MODEL)

print(f"Original Document: {documents[id]}\n\n")
print(f"Generated Queries: {querydoc}")

Original Document: {'uid': 't6Wc7OMks4U__B2__S61.88', 'text': " you look at, and who’s doing the telling. By studying this centuries-old screen we\xa0\nlearn about more than just the object itself. We learn about history — and how history,\xa0\xa0 like art, is not something we\xa0\ndiscover, but something we create. Hi! I'm Sarah Urist Green. I’m a curator and art educator, and\xa0\nthis is Crash Course Art History. You might think that art history\xa0\nis a stuffy subject that’s only\xa0\xa0 useful for clinching a trivia night victory. Or you might think it’s just\xa0\nmemorizing a bunch of names and dates,\xa0\xa0 with the occasional jump-scare\xa0\nfrom a creepy, long-limbed baby. But art history is really about thinking\xa0\ncritically by observing closely. It’s the study of objects and\xa0\nimages to understand the people,\xa0\xa0 places and time periods they come from. Art reflects different perspectives and stories\xa0\xa0 but also reveals the ideas that\xa0\nconnect people acro

In [100]:
# original llm response
querydoc

' [\n    "What are the fundamental approaches to studying art and how does it help us understand the objects and images we encounter?",\n    "How has the definition of art evolved throughout history and which cultural perspectives stand out in understanding different art forms?", \n    "Why is the story behind the Sistine Chapel ceiling so crucial in understanding the role of artists, patrons, and the shift towards Renaissance thought during this artistic era?", \n    "Who were some historical figures who shaped our current understanding of art through their writings on art history, like Plato, Aristotle, or Vasari? How has the history affected Western art as a whole?",\n     "How do we recognize artists and works that are considered to be classics in art history? What is the role and importance of a "canon of art history?"  Give an example of why some artistic traditions are left out of the canon". \n\n\n ]'

In [103]:
positions = find_symbol_positions(querydoc)
positions

{'[': [1],
 ']': [911],
 '"': [7, 131, 138, 275, 283, 460, 468, 671, 679, 806, 828, 904],
 "'": [],
 '-': [],
 '?': [130, 274, 459, 617, 670, 767, 827]}

In [102]:
extract_query5_v2(querydoc, positions)

DEBUG : quote_positions, single_quotes --  [7, 131, 138, 275, 283, 460, 468, 671, 679, 806, 828, 904] []


['What are the fundamental approaches to studying art and how does it help us understand the objects and images we encounter?',
 'How has the definition of art evolved throughout history and which cultural perspectives stand out in understanding different art forms?',
 'Why is the story behind the Sistine Chapel ceiling so crucial in understanding the role of artists, patrons, and the shift towards Renaissance thought during this artistic era?',
 'Who were some historical figures who shaped our current understanding of art through their writings on art history, like Plato, Aristotle, or Vasari? How has the history affected Western art as a whole?',
 'How do we recognize artists and works that are considered to be classics in art history? What is the role and importance of a',
 'Give an example of why some artistic traditions are left out of the canon']

### Generating Ground Truth File

In [117]:
df_query5 = generate_query_file(documents[:3], llm_model='gemma2:2b', retry=3)
df_query5.shape, df_query5.head(20)

INFO: initiated query generation
INFO: total text chunks 3


Generating Queries:   0%|          | 0/3 [00:00<?, ?it/s]

INFO: query generation finished
INFO: total documents - 3 | success - 2 | failure - 1


,uid,query
0,t6Wc7OMks4U__B2__S61.88,What exactly is art history?
1,t6Wc7OMks4U__B2__S61.88,How has the study of art changed over time?
2,t6Wc7OMks4U__B2__S61.88,How important is historical context?
3,t6Wc7OMks4U__B2__S61.88,Who decides which works of art should count to...
4,t6Wc7OMks4U__B2__S61.88,What are some modern examples of how the canon...
5,t6Wc7OMks4U__B3__S121.84,"economic upheaval, or personal/artistic prefer..."
6,t6Wc7OMks4U__B3__S121.84,How Do We Define It?


In [ ]:
# create the directory if it doesn't exist
os.makedirs("../data/ground_truth/", exist_ok=True)

export_df_to_json(df_query5, "../data/ground_truth/ground_truth_retrieval.json")